# Fase 2: Análisis de Calidad de Datos y Reglas de Negocio
Auditoría de integridad referencial, anomalías numéricas, coherencia cronológica y duplicados.

In [16]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# 1. Definir ruta y cargar las 11 fuentes
ruta_datos = Path("../data/raw/northwind")
archivos_csv = list(ruta_datos.glob("*.csv"))

tablas = {}
for archivo in archivos_csv:
    tablas[archivo.stem] = pd.read_csv(archivo)

print(f" Se cargaron exitosamente {len(tablas)} tablas para auditoría de calidad.")

 Se cargaron exitosamente 11 tablas para auditoría de calidad.


In [14]:
tablas["regions"]

,regionID,regionDescription
0,1,Eastern
1,2,Western
2,3,Northern
3,4,Southern


### 1 - Auditoría de Integridad Referencial (Claves Huérfanas)

In [ ]:

pruebas_integridad = [
    # (Tabla Origen, Columna FK, Tabla Destino, Columna PK)
    ("orders", "customerID", "customers", "customerID"),
    ("orders", "employeeID", "employees", "employeeID"),
    ("orders", "shipVia", "shippers", "shipperID"),
    ("order-details", "orderID", "orders", "orderID"),
    ("order-details", "productID", "products", "productID"),
    ("products", "supplierID", "suppliers", "supplierID"),
    ("products", "categoryID", "categories", "categoryID")
]

resultados_huerfanos = []

for t_origen, fk, t_destino, pk in pruebas_integridad:
    df_origen = tablas[t_origen]
    df_destino = tablas[t_destino]
    
    # Extraer valores únicos de la clave primaria válida
    claves_validas = set(df_destino[pk].dropna())
    
    # Filtrar registros en origen cuya FK no esté en las PK válidas
    huerfanos = df_origen[~df_origen[fk].isin(claves_validas) & df_origen[fk].notna()]
    conteo_huerfanos = len(huerfanos)
    
    resultados_huerfanos.append({
        "relacion": f"{t_origen}.{fk} -> {t_destino}.{pk}",
        "total_registros_origen": len(df_origen),
        "claves_huerfanas_encontradas": conteo_huerfanos,
        "estado": "[OK] Válida" if conteo_huerfanos == 0 else "[ERROR] Inconsistente"
    })

df_resumen_integridad = pd.DataFrame(resultados_huerfanos)
display(df_resumen_integridad)


,relacion,total_registros_origen,claves_huerfanas_encontradas,estado
0,orders.customerID -> customers.customerID,830,0,[OK] Válida
1,orders.employeeID -> employees.employeeID,830,0,[OK] Válida
2,orders.shipVia -> shippers.shipperID,830,0,[OK] Válida
3,order-details.orderID -> orders.orderID,2155,0,[OK] Válida
4,order-details.productID -> products.productID,2155,0,[OK] Válida
5,products.supplierID -> suppliers.supplierID,77,0,[OK] Válida
6,products.categoryID -> categories.categoryID,77,0,[OK] Válida


### 2 - Auditoría de Reglas Numéricas (Cantidades, Precios, Descuentos)

In [19]:
df_od = tablas["order-details"]
df_ord = tablas["orders"]
df_prod = tablas["products"]
reglas_numericas = [
    {
        "tabla": "order-details",
        "campo": "unitPrice",
        "regla": "Precio unitario debe ser > 0",
        "anomalias": (df_od["unitPrice"] <= 0).sum(),
        "minimo": df_od["unitPrice"].min(),
        "maximo": df_od["unitPrice"].max()
    },
    {
        "tabla": "order-details",
        "campo": "quantity",
        "regla": "Cantidad vendida debe ser > 0",
        "anomalias": (df_od["quantity"] <= 0).sum(),
        "minimo": df_od["quantity"].min(),
        "maximo": df_od["quantity"].max()
    },
    {
        "tabla": "order-details",
        "campo": "discount",
        "regla": "Descuento debe estar entre 0.0 y 1.0",
        "anomalias": ((df_od["discount"] < 0) | (df_od["discount"] > 1)).sum(),
        "minimo": df_od["discount"].min(),
        "maximo": df_od["discount"].max()
    },
    {
        "tabla": "orders",
        "campo": "freight",
        "regla": "Flete / envío debe ser >= 0",
        "anomalias": (df_ord["freight"] < 0).sum(),
        "minimo": df_ord["freight"].min(),
        "maximo": df_ord["freight"].max()
    },
    {
        "tabla": "products",
        "campo": "unitPrice",
        "regla": "Precio de catálogo debe ser > 0",
        "anomalias": (df_prod["unitPrice"] <= 0).sum(),
        "minimo": df_prod["unitPrice"].min(),
        "maximo": df_prod["unitPrice"].max()
    },
    {
        "tabla": "products",
        "campo": "unitsInStock",
        "regla": "Stock en almacén debe ser >= 0",
        "anomalias": (df_prod["unitsInStock"] < 0).sum(),
        "minimo": df_prod["unitsInStock"].min(),
        "maximo": df_prod["unitsInStock"].max()
    }
]
df_resumen_reglas = pd.DataFrame(reglas_numericas)
df_resumen_reglas["estado"] = df_resumen_reglas["anomalias"].apply(
    lambda x: "[OK] Conforme" if x == 0 else f"[ERROR] {x} Anomalías"
)
display(df_resumen_reglas)

,tabla,campo,regla,anomalias,minimo,maximo,estado
0,order-details,unitPrice,Precio unitario debe ser > 0,0,2.00,263.50,[OK] Conforme
1,order-details,quantity,Cantidad vendida debe ser > 0,0,1.00,130.00,[OK] Conforme
2,order-details,discount,Descuento debe estar entre 0.0 y 1.0,0,0.00,0.25,[OK] Conforme
3,orders,freight,Flete / envío debe ser >= 0,0,0.02,1007.64,[OK] Conforme
4,products,unitPrice,Precio de catálogo debe ser > 0,0,2.50,263.50,[OK] Conforme
5,products,unitsInStock,Stock en almacén debe ser >= 0,0,0.00,125.00,[OK] Conforme


### 3 - Auditoría Cronológica de Pedidos y Envíos

In [20]:
df_orders_dates = tablas["orders"].copy()
# Conversión temporal a datetime para evaluar diferencias temporales
df_orders_dates["orderDate"] = pd.to_datetime(df_orders_dates["orderDate"])
df_orders_dates["requiredDate"] = pd.to_datetime(df_orders_dates["requiredDate"])
df_orders_dates["shippedDate"] = pd.to_datetime(df_orders_dates["shippedDate"])

In [22]:
df_orders_dates["requiredDate"].dtypes

dtype('<M8[us]')

In [25]:
# Filtrar solo órdenes que ya fueron despachadas (shippedDate no nulo)
despachadas = df_orders_dates[df_orders_dates["shippedDate"].notna()]
total_despachadas = len(despachadas)
total_despachadas

809

In [26]:
# 1. Regla: Despacho anterior a la orden (Anomalía crítica)
despacho_imposible = (despachadas["shippedDate"] < despachadas["orderDate"]).sum()
despacho_imposible

np.int64(0)

In [28]:
# 2. Regla: Requerimiento anterior a la orden (Anomalía crítica)
requerimiento_imposible = (df_orders_dates["requiredDate"] < df_orders_dates["orderDate"]).sum()
requerimiento_imposible

np.int64(0)

In [29]:
# 3. Métrica de negocio: Envíos tardíos (KPI logístico)
envios_tardios = (despachadas["shippedDate"] > despachadas["requiredDate"]).sum()
pct_tardios = (envios_tardios / total_despachadas * 100)
auditoria_fechas = pd.DataFrame([
    {
        "prueba": "Despacho anterior a orden (shippedDate < orderDate)",
        "tipo": "Error de Calidad",
        "casos": despacho_imposible,
        "interpretacion": "[OK] Conforme (0 errores)" if despacho_imposible == 0 else "[ERROR] Anomalía detectada"
    },
    {
        "prueba": "Requerimiento anterior a orden (requiredDate < orderDate)",
        "tipo": "Error de Calidad",
        "casos": requerimiento_imposible,
        "interpretacion": "[OK] Conforme (0 errores)" if requerimiento_imposible == 0 else "[ERROR] Anomalía detectada"
    },
    {
        "prueba": "Envíos con retraso (shippedDate > requiredDate)",
        "tipo": "Métrica de Negocio (KPI)",
        "casos": envios_tardios,
        "interpretacion": f"ℹ️ {envios_tardios} pedidos demorados ({pct_tardios:.2f}% de las órdenes)"
    }
])
display(auditoria_fechas)

,prueba,tipo,casos,interpretacion
0,Despacho anterior a orden (shippedDate < order...,Error de Calidad,0,[OK] Conforme (0 errores)
1,Requerimiento anterior a orden (requiredDate <...,Error de Calidad,0,[OK] Conforme (0 errores)
2,Envíos con retraso (shippedDate > requiredDate),Métrica de Negocio (KPI),37,ℹ️ 37 pedidos demorados (4.57% de las órdenes)


### 4 - Auditoría de Duplicados Completos a Nivel de Fila

In [30]:
resumen_duplicados = []
for nombre_tabla, df in tablas.items():
    total_filas = len(df)
    filas_duplicadas = df.duplicated().sum()
    pct_duplicados = (filas_duplicadas / total_filas * 100) if total_filas > 0 else 0.0
    
    resumen_duplicados.append({
        "tabla": nombre_tabla,
        "total_filas": total_filas,
        "filas_duplicadas": filas_duplicadas,
        "pct_duplicados": round(pct_duplicados, 2),
        "estado": "[OK] Sin duplicados" if filas_duplicadas == 0 else f"[ADVERTENCIA] {filas_duplicadas} Duplicados"
    })
df_resumen_duplicados = pd.DataFrame(resumen_duplicados)
display(df_resumen_duplicados)


,tabla,total_filas,filas_duplicadas,pct_duplicados,estado
0,categories,8,0,0.0,[OK] Sin duplicados
1,customers,91,0,0.0,[OK] Sin duplicados
2,employee-territories,49,0,0.0,[OK] Sin duplicados
3,employees,9,0,0.0,[OK] Sin duplicados
4,order-details,2155,0,0.0,[OK] Sin duplicados
5,orders,830,0,0.0,[OK] Sin duplicados
6,products,77,0,0.0,[OK] Sin duplicados
7,regions,4,0,0.0,[OK] Sin duplicados
8,shippers,3,0,0.0,[OK] Sin duplicados
9,suppliers,29,0,0.0,[OK] Sin duplicados


### 5 - Conclusiones de la Auditoría de Calidad (Fase 2)

1. **Integridad Referencial:** Perfecta. Todas las claves foráneas de órdenes, detalles, clientes, productos y proveedores tienen una clave primaria válida de destino (0 huérfanos).
2. **Consistencia Financiera y Cuantitativa:** No existen valores negativos en importes, precios ni cantidades. Los descuentos están acotados comercialmente entre 0.0 y 0.25 (máximo 25%).
3. **Comportamiento Operativo:** No existen fechas incongruentes (despachos previos a la orden). Se identificaron 37 pedidos con despacho posterior a la fecha requerida (4.57%), estableciendo una tasa de entrega a tiempo del 95.43%.
4. **Duplicidad:** Se certifica que no existen filas redundantes ni duplicadas en ninguna de las 11 fuentes.
